# ABAA witness-junction session, Chalmers protocol — Keithley 2450

Protocol after Križan, et al. arXiv: 2602.20002, at room temperature:

- **Iteration** = a burst of $M = 6$ bipolar pulses ($+V_A$, blank, $-V_A$, blank), then an **I-V readout**: sweep $-13 \to +13$ mV and take $R$ from a linear fit.
- **Bursts are timed by the instrument** (a source-list sweep), so pulse width does not depend on the PC.
- **Reference point**: $R_\mathrm{Total} = R(t_\mathrm{stop} + 30\ \mathrm{min})$. The stop rule comes from calibrating $R_\mathrm{Total}$ against $R_\mathrm{Active}$.
- **Large changes** (more than about 10 %) use the stepped protocol: $+0.10\,R_0$ per step, then 3 h of relaxation.

**Pulse timing** (Križan Fig. 1a): each polarity is held for one 200 ms bias cycle, followed by 100 ms blanking. One bipolar pulse is 0.6 s, a burst of 6 is 3.6 s, and one iteration including the I-V readout takes about 4–5 s.

**This is unverified on hardware.** The SCPI list-sweep calls follow the 2450 reference manual but have not been run on an instrument. Before any junction:
1. Run sections 1–5.
2. Put a **5 kΩ resistor** on the probes and run section 5b. It checks the burst timing, prints the *measured* pulse and blank widths from the instrument timestamps, and checks that the I-V fit reads 5 kΩ.
3. Land probes only with the output **on at 0 V**. Every routine returns to 0 V when it finishes, aborts, or is interrupted.

**Allocation for 10 JJs**:
- JJ01–02: breakdown (+ / −).
- JJ03–05: rate vs amplitude; each tuned +10 %, then a 30 min hold.
- JJ06: +10 %, then 24 h relaxation.
- JJ07–08: targeting trial with the calibrated stop rule.
- JJ09–10: untouched controls.

## 1. Settings

In [1]:
import time, json, datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import pyvisa
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

In [2]:
# ---- edit these ----
RESOURCE  = "TCPIP0::192.168.50.2::inst0::INSTR"   # pyvisa.ResourceManager().list_resources()
DATA_DIR  = Path(r"C:\Users\hatlab\Probing_Data\Israa")
SAMPLE    = "witness_batch_01"
ROOM_T_C  = None          # chuck / room temperature in C, by hand
FOUR_WIRE = False         # True: two needles per pad, sense leads on the second pair

In [3]:
# burst (Križan defaults)
M_PULSES  = 6             # bipolar pulses per iteration
T_PULSE   = 0.200         # s at +V_A and at -V_A (one 200 ms bias cycle, Križan Fig. 1a)
T_BLANK   = 0.100         # s at 0 V after each polarity (100 ms blanking)
DT_POINT  = 0.025         # s per list point; T_PULSE and T_BLANK must be multiples of it
NPLC_BURST = 0.05         # fast current reading inside the burst (protection only)

In [4]:
# I-V readout
V_IV      = 0.013         # sweep -V_IV .. +V_IV
N_IV      = 27            # points (1 mV steps)
NPLC_IV   = 1.0
DELAY_IV  = 0.005

R_SHORT   = 260.0         # Ohm: below this a junction counts as shorted (Križan)

DATA_DIR.mkdir(exist_ok=True)

## 2. Instrument

In [32]:

I_RANGES = [1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]

def pick_range(i):
    return next(r for r in I_RANGES if r >= abs(i))


class K2450:
    def __init__(self, resource, terminals):
        self.rm = pyvisa.ResourceManager()
        self.inst = self.rm.open_resource(resource)
        self.inst.timeout = 30_000
        self.terminals = terminals      # "FRON" or "REAR"
        self.inst.write_termination = "\n"
        self.inst.read_termination = "\n"
        self.ilim = None
        self.timing_reported = False
        print(self.q("*IDN?"))

    def w(self, cmd):
        self.inst.write(cmd)

    def q(self, cmd):
        return self.inst.query(cmd).strip()

    # ---- basic state ----
    def configure(self, ilim=1e-3, four_wire=FOUR_WIRE):
        self.w("*CLS")
        self.w("*RST")
        self.w(f":ROUT:TERM {self.terminals}")
        self.w(":SOUR:FUNC VOLT")
        self.w(":SOUR:VOLT:RANG 2")
        self.w(":SOUR:VOLT:READ:BACK ON")
        self.w(':SENS:FUNC "CURR"')
        self.w(f":SENS:CURR:RSEN {'ON' if four_wire else 'OFF'}")
        self.w(":OUTP:VOLT:SMOD NORM")       # output-off state: 0 V
        self.set_ilim(ilim)
        self.set_v(0.0)
        self.check_errors()

    def set_ilim(self, ilim):
        self.ilim = float(ilim)
        self.w(f":SOUR:VOLT:ILIM {self.ilim:.6g}")

    def set_v(self, v):
        self.w(f":SOUR:VOLT {v:.6f}")

    def set_range(self, r):
        self.w(f":SOUR:VOLT:RANG {r}")

    def output(self, on):
        self.w(":OUTP ON" if on else ":OUTP OFF")

    def read(self):
        i, v = (float(x) for x in self.q(':READ? "defbuffer1", READ, SOUR').split(","))
        return v, i

    def in_compliance(self, i):
        return abs(i) >= 0.98 * self.ilim

    def check_errors(self):
        errs = []
        for _ in range(50):
            e = self.q(":SYST:ERR?")
            if e.startswith("0,"):
                break
            errs.append(e)
        if errs:
            print("SMU errors:", errs)
        return errs

    def safe(self):
        try:
            self.w(":ABOR")
            self.set_v(0.0)
            self.set_range(2)
        except Exception as ex:
            print("COULD NOT ZERO OUTPUT:", ex)

    def close(self):
        self.safe()
        self.output(False)
        self.inst.close()

    # ---- instrument-timed list sweep ----
    def _list_sweep(self, levels, delay):
        self.w(':TRAC:CLE "defbuffer1"')

        # first: create the voltage list
        for k in range(0, len(levels), 100):
            cmd = ":SOUR:LIST:VOLT" if k == 0 else ":SOUR:LIST:VOLT:APP"
            self.w(f"{cmd} " + ",".join(f"{v:.5f}" for v in levels[k:k + 100]))

    # second: set up the sweep that uses it
        self.w(f':SOUR:SWE:VOLT:LIST 1, {delay:.4f}, 1, ON, "defbuffer1"')
        errs = self.check_errors()
        if errs:
            raise RuntimeError(f"Sweep setup failed: {errs}")

    # third: run it and read the data
        self.w(":INIT")
        self.q("*OPC?")
        n = int(self.q(':TRAC:ACT? "defbuffer1"'))
        if n == 0:
            self.set_v(0.0)
            self.output(True)
            errs = self.check_errors()
            raise RuntimeError(f"Sweep gave no readings: current limit on the first point "
                                     f"(short?). {errs}")
        raw = self.q(f':TRAC:DATA? 1, {n}, "defbuffer1", READ, SOUR, REL').split(",")
        self.set_v(0.0)
        self.output(True)   # the sweep ends with the output off; go back to on at 0 V
        a = np.array(raw, float).reshape(-1, 3)
        return pd.DataFrame({"I": a[:, 0], "V": a[:, 1], "t": a[:, 2]}), n < len(levels)

    def burst(self, V_A, M=M_PULSES, t_pulse=T_PULSE, t_blank=T_BLANK, dt_point=DT_POINT):
        n_p, n_b = round(t_pulse / dt_point), round(t_blank / dt_point)
        assert n_p >= 1 and n_b >= 1, "T_PULSE and T_BLANK must be >= DT_POINT"
        levels = ([V_A] * n_p + [0.0] * n_b + [-V_A] * n_p + [0.0] * n_b) * M
        t_meas = NPLC_BURST / 60
        delay = max(dt_point - t_meas - 0.0052, 0.0)
        self.set_range(2)
        self.w(f":SENS:CURR:NPLC {NPLC_BURST}")
        self.w(f":SENS:CURR:RANG {pick_range(self.ilim)}")
        d, aborted = self._list_sweep(levels, delay)
        pos, neg = d[d.V > V_A / 2], d[d.V < -V_A / 2]
        out = dict(I_pos=pos.I.abs().max(), I_neg=neg.I.abs().max(),
                   compliance=aborted or self.in_compliance(d.I.abs().max()),
                   t_point=float(np.median(np.diff(d.t))) if len(d) > 1 else np.nan)
        out["t_pulse_meas"] = out["t_point"] * n_p
        out["t_blank_meas"] = out["t_point"] * n_b
        if not self.timing_reported:
            print(f"  measured burst timing: {out['t_point'] * 1e3:.1f} ms/point -> pulse "
                  f"{out['t_pulse_meas'] * 1e3:.0f} ms, blank {out['t_blank_meas'] * 1e3:.0f} ms "
                  f"(asked {t_pulse * 1e3:.0f} / {t_blank * 1e3:.0f})")
            self.timing_reported = True
        return out

    def iv(self, v=V_IV, n=N_IV):
        """I-V sweep on the 200 mV range; R from a linear fit (intercept absorbs offsets)."""
        self.set_v(0.0)
        self.set_range(0.2)
        self.w(f":SENS:CURR:NPLC {NPLC_IV}")
        self.w(":SENS:CURR:RANG:AUTO ON")
        d, _ = self._list_sweep(list(np.linspace(-v, v, n)), DELAY_IV)
        self.set_range(2)
        slope, icpt = np.polyfit(d.V, d.I, 1)
        resid = d.I - (slope * d.V + icpt)
        R = 1.0 / slope
        return R, float(resid.std() * R / v)   # R, relative fit residual

## 3. Logging

In [33]:
EVENT = {"iv": 0, "burst": 1, "ramp": 2}


class RunLog:
    """Crash-safe CSV row by row; JSON for parameters; ddh5 export at the end."""
    cols = ["t", "event", "iter", "V", "I", "R", "fit_res"]

    def __init__(self, jj, kind, **meta):
        stamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
        self.name = f"{stamp}_{SAMPLE}_{jj}_{kind}"
        self.csv = DATA_DIR / f"{self.name}.csv"
        self.meta = dict(sample=SAMPLE, jj=jj, kind=kind, start=stamp, room_T_C=ROOM_T_C,
                         four_wire=FOUR_WIRE, M=M_PULSES, t_pulse=T_PULSE, t_blank=T_BLANK,
                         V_IV=V_IV, N_IV=N_IV, NPLC_IV=NPLC_IV, **meta)
        self.save_meta()
        self.t0 = time.time()
        self.rows = []
        with open(self.csv, "w") as f:
            f.write(",".join(self.cols) + "\n")

    def save_meta(self):
        (DATA_DIR / f"{self.name}.json").write_text(json.dumps(self.meta, indent=2, default=str))

    def add(self, event, it, V=np.nan, I=np.nan, R=np.nan, fit_res=np.nan):
        row = [time.time() - self.t0, event, it, V, I, R, fit_res]
        self.rows.append(row)
        with open(self.csv, "a") as f:
            f.write(",".join(str(x) for x in row) + "\n")

    @property
    def df(self):
        return pd.DataFrame(self.rows, columns=self.cols)

    def finish(self, **more_meta):
        self.meta.update(more_meta)
        self.save_meta()
        try:
            from labcore.data.datadict import DataDict
            from labcore.data.datadict_storage import datadict_to_hdf5
            d = self.df
            dd = DataDict(
                t=dict(values=d.t.values, unit="s"),
                event=dict(values=d.event.map(EVENT).values, axes=["t"]),
                iter=dict(values=d.iter.values, axes=["t"]),
                V=dict(values=d.V.values, axes=["t"], unit="V"),
                I=dict(values=d.I.values, axes=["t"], unit="A"),
                R=dict(values=d.R.values, axes=["t"], unit="Ohm"),
                fit_res=dict(values=d.fit_res.values, axes=["t"]),
            )
            for k, v in self.meta.items():
                dd.add_meta(k, str(v))
            dd.validate()
            datadict_to_hdf5(dd, str(DATA_DIR / self.name))
        except Exception as ex:
            print(f"ddh5 export skipped ({ex!r}); CSV is complete: {self.csv}")

## 4. Routines

| Routine | What it does |
|---|---|
| `measure_R(jj)` | One I-V readout. Lift, re-land and repeat to get contact repeatability. |
| `breakdown_ramp` | Steps $\lvert V\rvert$ up until the current limit, or a sudden drop in $V/I$. Confirms with an I-V: below `R_SHORT` it is shorted. |
| `tune` | Repeats burst → I-V until `R_stop`, `target_frac`, or `t_max`. Aborts on current limit or $R <$ `R_SHORT`. |
| `hold` | I-V every `t_probe` s, with time measured from the last pulse. Returns $R$ at 30 min. |
| `tune_and_hold` | `tune`, then a 30 min `hold`. Gives one ($R_\mathrm{Active}$, $R_\mathrm{Total}$) calibration pair. |
| `stepped` | Križan's stepped protocol for large changes. |

In [34]:
BASE = {}

def measure_R(jj):
    log = RunLog(jj, "baseline")
    R, res = smu.iv()
    log.add("iv", 0, R=R, fit_res=res)
    log.finish(R=R)
    BASE.setdefault(jj, []).append(R)
    msg = f"{jj}: R = {R:.2f} Ohm  (fit residual {res * 100:.3f} %)"
    if len(BASE[jj]) > 1:
        r = np.array(BASE[jj])
        msg += f"   n = {len(r)}, re-land spread {np.ptp(r) / r.mean() * 100:.3f} %"
    print(msg)
    return R


def breakdown_ramp(jj, polarity=+1, v_step=0.01, v_max=2.0, dwell=0.2, ilim=1e-3, jump_frac=0.3):
    log = RunLog(jj, f"bdv_{'pos' if polarity > 0 else 'neg'}",
                 polarity=polarity, v_step=v_step, v_max=v_max, dwell=dwell, ilim=ilim)
    smu.set_ilim(ilim)
    R0, _ = smu.iv()
    log.add("iv", 0, R=R0)
    smu.w(":SENS:CURR:NPLC 1")
    smu.w(":SENS:CURR:RANG:AUTO ON")
    smu.set_range(2)
    V_bd, R_prev = None, None
    try:
        for k, v in enumerate(np.arange(v_step, v_max + v_step / 2, v_step)):
            smu.set_v(polarity * v)
            time.sleep(dwell)
            Vm, I = smu.read()
            R = Vm / I
            log.add("ramp", k, V=Vm, I=I, R=R)
            if smu.in_compliance(I) or (R_prev is not None and R < (1 - jump_frac) * R_prev):
                V_bd = float(v)
                break
            R_prev = R
    finally:
        smu.safe()
    R1, _ = smu.iv()
    log.add("iv", 1, R=R1)
    shorted = R1 < R_SHORT
    log.finish(R0=R0, R_after=R1, V_bd=V_bd, shorted=shorted)
    if V_bd is None:
        print(f"{jj}: no breakdown up to {v_max:.2f} V   R {R0:.1f} -> {R1:.1f} Ohm")
    else:
        print(f"{jj}: stop at {polarity * V_bd:+.3f} V   R {R0:.1f} -> {R1:.1f} Ohm   "
              + ("SHORTED" if shorted else "NOT shorted: check the I-V plot before trusting V_bd"))
    return V_bd, log


def tune(jj, V_A, t_max=1800, target_frac=None, R_stop=None, ilim_factor=3.0, jump_frac=0.02,
         print_every=20, R0=None):
    log = RunLog(jj, "tune", V_A=V_A, t_max=t_max, target_frac=target_frac, R_stop=R_stop,
                 ilim_factor=ilim_factor)
    R_first, res = smu.iv()
    log.add("iv", 0, R=R_first, fit_res=res)
    R0 = R_first if R0 is None else R0            # stepped runs keep the original R(0)
    smu.set_ilim(min(ilim_factor * V_A / R_first, 1e-2))
    print(f"{jj}: R = {R_first:.2f} Ohm (R0 = {R0:.2f})   V_A = {V_A:.3f} V   "
          f"I_lim = {smu.ilim * 1e6:.0f} uA")
    R, R_prev, n, reason = R_first, R_first, 0, "time cap"
    t_start = t_stop = time.time()
    try:
        while True:
            n += 1
            b = smu.burst(V_A)
            t_stop = time.time()
            log.add("burst", n, V=V_A, I=max(b["I_pos"], b["I_neg"]))
            if b["compliance"]:
                raise RuntimeError(f"current limit during burst (I+ {b['I_pos']:.2e}, I- {b['I_neg']:.2e} A)")
            R, res = smu.iv()
            log.add("iv", n, R=R, fit_res=res)
            if R < R_SHORT:
                raise RuntimeError(f"R = {R:.1f} Ohm: shorted")
            if abs(R - R_prev) / R_prev > jump_frac:
                print(f"  jump at iteration {n}: {R_prev:.1f} -> {R:.1f} Ohm")
            R_prev = R
            if n % print_every == 0:
                print(f"  it {n:5d}   t = {t_stop - t_start:6.0f} s   R = {R:9.2f} Ohm   "
                      f"dR/R0 = {(R / R0 - 1) * 100:6.2f} %")
            if R_stop is not None and R >= R_stop:
                reason = "R_stop reached"; break
            if target_frac is not None and R / R0 - 1 >= target_frac:
                reason = "target reached"; break
            if t_stop - t_start >= t_max:
                break
    except KeyboardInterrupt:
        reason = "interrupted"
    except RuntimeError as ex:
        reason = f"ABORT: {ex}"
    finally:
        smu.safe()
    res = dict(jj=jj, V_A=V_A, R0=R0, R_start=R_first, R_active=R, dR_active=R / R0 - 1,
               iterations=n, pulses=n * M_PULSES, duration_s=t_stop - t_start,
               t_stop=t_stop, reason=reason, csv=str(log.csv))
    log.finish(**{k: v for k, v in res.items() if k != "csv"})
    print(f"{jj}: {reason}.  {n} iterations, {res['duration_s']:.0f} s,  "
          f"R {R_first:.2f} -> {R:.2f} Ohm  (dR/R0 = {res['dR_active'] * 100:+.2f} %)")
    return res


def hold(jj, t_stop, R0, t_max=1800, t_probe=60, print_every=10):
    """I-V every t_probe s; t measured from the last pulse. Returns (log, R at 30 min)."""
    log = RunLog(jj, "hold", t_max=t_max, t_probe=t_probe, R0=R0)
    log.t0 = t_stop
    k = 0
    try:
        while True:                          # last read is at or after t_max
            t_k = time.time()
            R, res = smu.iv()
            log.add("iv", k, R=R, fit_res=res)
            if k % print_every == 0:
                print(f"  t = {(t_k - t_stop) / 60:7.1f} min   R = {R:9.2f} Ohm   "
                      f"dR/R0 = {(R / R0 - 1) * 100:+.3f} %")
            k += 1
            if t_k - t_stop >= t_max:
                break
            time.sleep(max(t_probe - (time.time() - t_k), 0))
    except KeyboardInterrupt:
        pass
    finally:
        smu.safe()
    d = log.df[log.df.event == "iv"]
    R30 = float(np.interp(1800, d.t, d.R)) if len(d) and d.t.max() >= 1800 else np.nan
    log.finish(n_reads=k, R_30min=R30)
    return log, R30


CAL = []   # (jj, V_A, R0, R_active, R_30min) calibration pairs for the stop rule

def tune_and_hold(jj, V_A, target_frac=0.10, t_max=1800, t_hold=1800, **kw):
    r = tune(jj, V_A, t_max=t_max, target_frac=target_frac, **kw)
    if r["reason"].startswith("ABORT"):
        return r
    lg, R30 = hold(jj, r["t_stop"], r["R0"], t_max=t_hold)
    r.update(R_30min=R30, dR_total=R30 / r["R0"] - 1, hold_csv=str(lg.csv))
    if np.isfinite(R30):
        CAL.append(dict(jj=jj, V_A=V_A, R0=r["R0"], R_active=r["R_active"], R_30min=R30))
        print(f"{jj}: R(30 min) = {R30:.2f} Ohm   relaxation = "
              f"{(R30 - r['R_active']) / r['R0'] * 100:+.2f} % of R0")
    return r


def stepped(jj, V_A, step_frac=0.10, n_steps=3, t_relax=3 * 3600, t_probe=300, t_max_step=3600):
    """Križan's stepped protocol: each step adds step_frac * R(0) actively, then relaxes."""
    R0, _ = smu.iv()
    out = []
    for k in range(1, n_steps + 1):
        print(f"--- step {k}/{n_steps}: active target {k * step_frac * 100:.0f} % above R0")
        r = tune(f"{jj}_s{k}", V_A, t_max=t_max_step, R_stop=R0 * (1 + k * step_frac), R0=R0)
        out.append(r)
        if r["reason"].startswith("ABORT"):
            break
        hold(f"{jj}_s{k}", r["t_stop"], R0, t_max=t_relax, t_probe=t_probe, print_every=6)
    return out

In [35]:
TERMINALS = 'REAR'

## 5. Connect and arm (output on at 0 V)

In [36]:
# smu = K2450(RESOURCE)
# smu.configure(ilim=1e-3)
# smu.set_v(0.0)
# smu.output(True)
# print("Output ON at 0 V.")

try:
    smu.close()
except Exception:
    pass

smu = K2450(RESOURCE, terminals=TERMINALS)
smu.configure(ilim=1e-3)
smu.set_v(0.0)

KEITHLEY INSTRUMENTS,MODEL 2450,MYFP002618,1.7.16a


## 5b. Dummy-resistor check (5 kΩ on the probes — **not a junction**)
Expect $R \approx 5$ kΩ, both before and after the bursts, and measured pulse/blank widths close to the requested ones. If the timing is off, adjust `DT_POINT` or `NPLC_BURST`. Watch for `SMU errors:`.

In [37]:
smu.timing_reported = False
measure_R("DUMMY")
smu.set_ilim(1e-3)
b = smu.burst(0.5, M=2)
print(b)
measure_R("DUMMY")
smu.check_errors()

DUMMY: R = 250.63 Ohm  (fit residual 0.048 %)
  measured burst timing: nan ms/point -> pulse nan ms, blank nan ms (asked 200 / 100)
{'I_pos': 0.0009999949, 'I_neg': nan, 'compliance': True, 't_point': nan, 't_pulse_meas': nan, 't_blank_meas': nan}
DUMMY: R = 250.36 Ohm  (fit residual 0.055 %)   n = 2, re-land spread 0.107 %


[]

In [38]:
smu.output(True)
smu.burst(0.3, M=1)
print("Output:", smu.q(":OUTP?"), " Set V:", smu.q(":SOUR:VOLT?"))
smu.output(False)

Output: 1  Set V: 0


In [39]:
smu.burst(0.1, M=1)

{'I_pos': 0.0004010847,
 'I_neg': 0.0004007079,
 'compliance': False,
 't_point': 0.02501730000000002,
 't_pulse_meas': 0.20013840000000016,
 't_blank_meas': 0.10006920000000008}

In [40]:
smu.iv()

(250.30424714259155, 0.0005142549471806459)

## 6. Baseline — all junctions
Land, measure, lift, re-land, measure again. The re-land spread is the floor on any tuning precision.

In [41]:
measure_R("JJ01")

JJ01: R = 250.18 Ohm  (fit residual 0.044 %)


250.18117230180334

In [42]:
pd.DataFrame({jj: pd.Series(r) for jj, r in BASE.items()}).T.assign(
    mean=lambda d: d.mean(axis=1),
    spread_pct=lambda d: (d.max(axis=1) - d.min(axis=1)) / d.mean(axis=1) * 100)

,0,1,mean,spread_pct
DUMMY,250.633346,250.364261,250.498804,0.10742
JJ01,250.181172,NaN,250.181172,0.00000


## 7. Breakdown voltage — JJ01 (+), JJ02 (−)
Destroys the junction by design.

In [43]:
V_bd1, log_bd1 = breakdown_ramp("JJ01", polarity=+1)

JJ01: stop at +0.250 V   R 250.3 -> 249.7 Ohm   SHORTED


In [44]:
V_bd2, log_bd2 = breakdown_ramp("JJ02", polarity=-1)

JJ02: stop at -0.250 V   R 250.5 -> 249.4 Ohm   SHORTED


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for lg, lab in ((log_bd1, "JJ01 +"), (log_bd2, "JJ02 −")):
    d = lg.df[lg.df.event == "ramp"]
    ax[0].plot(d.V.abs(), d.I.abs() * 1e6, ".-", label=lab)
    ax[1].plot(d.V.abs(), d.R, ".-", label=lab)
ax[0].set(xlabel="|V| (V)", ylabel="|I| (uA)")
ax[1].set(xlabel="|V| (V)", ylabel="V/I (Ohm)")
for a in ax: a.legend()
plt.tight_layout()

V_BD = min(v for v in (V_bd1, V_bd2) if v is not None)
print(f"V_BD = {V_BD:.3f} V   (Križan: ~1.1 V and V_A = 0.85 V, i.e. ~0.77 V_BD)")

## 8. Rate vs amplitude + stop-rule calibration — JJ03–05
Each run tunes $+10\,\%$ (30 min cap), then holds 30 min, so each gives a rate **and** one ($R_\mathrm{Active}$, $R_\mathrm{Total}$) pair. Lowest amplitude first.

In [ ]:
V_A_LIST = {"JJ03": 0.70 * V_BD, "JJ04": 0.75 * V_BD, "JJ05": 0.80 * V_BD}
RESULTS = {}

In [ ]:
RESULTS["JJ03"] = tune_and_hold("JJ03", V_A_LIST["JJ03"])

In [ ]:
RESULTS["JJ04"] = tune_and_hold("JJ04", V_A_LIST["JJ04"])

In [ ]:
RESULTS["JJ05"] = tune_and_hold("JJ05", V_A_LIST["JJ05"])

## 9. JJ06 — +10 %, then 24 h relaxation
Leave the probes landed, disable sleep, and keep this kernel running. Križan's relaxation had not settled after 24 h.

In [ ]:
V_A_RUN = V_A_LIST["JJ04"]           # pick from section 10a: fastest amplitude that is clearly safe
r6 = RESULTS["JJ06"] = tune("JJ06", V_A_RUN, target_frac=0.10, t_max=3600)
if not r6["reason"].startswith("ABORT"):
    log_relax6, R30_6 = hold("JJ06", r6["t_stop"], r6["R0"], t_max=24 * 3600, t_probe=120)
    CAL.append(dict(jj="JJ06", V_A=V_A_RUN, R0=r6["R0"], R_active=r6["R_active"], R_30min=R30_6))

## 10. Analysis

**a. Rate.** Fit the active run as $\Delta R/R_0 = a\,t + b\,t^2$, as Križan does, and compare amplitudes with $a(V) = a_0\, e^{V/V_0}$. They found $V_0 = 55$–$91$ mV.

**b. Stop rule.** Fit $\Delta R_\mathrm{Total}/R_0 = s\,\Delta R_\mathrm{Active}/R_0 + \delta$ across the calibration pairs. Križan found $\delta = 3.29 \pm 0.19\,\%$ and $s = 1.00$–$1.13$. To land on $R_T$, stop at
$$R_\mathrm{stop} = R_0\left[1 + \frac{R_T/R_0 - 1 - \delta}{s}\right].$$
The scatter of the fit residuals is the precision you can expect.

**c. Relaxation.** Plot JJ06 against $\log t$ (Križan: logarithmic in time).

In [ ]:
def fit_active(csv):
    d = pd.read_csv(csv)
    d = d[d.event == "iv"]
    t = (d.t - d.t.iloc[0]).values
    y = d.R.values / d.R.values[0] - 1
    (a, b), _ = curve_fit(lambda t, a, b: a * t + b * t ** 2, t, y, p0=(1e-4, 0.0))
    return dict(rate_pct_per_min=a * 6000, b=b, t=t, y=y, f=lambda tt: a * tt + b * tt ** 2)


fig, ax = plt.subplots(1, 2, figsize=(11, 4))
rows = []
for jj, r in RESULTS.items():
    if r["reason"].startswith("ABORT"):
        print(f"{jj}: {r['reason']}")
        continue
    fa = fit_active(r["csv"])
    ax[0].plot(fa["t"] / 60, fa["y"] * 100, ".", ms=3)
    ax[0].plot(fa["t"] / 60, fa["f"](fa["t"]) * 100, "-", label=f"{jj}, {r['V_A']:.3f} V")
    rows.append(dict(jj=jj, V_A=r["V_A"], V_A_over_VBD=r["V_A"] / V_BD,
                     rate_pct_per_min=fa["rate_pct_per_min"], pulses=r["pulses"],
                     dR_active_pct=r["dR_active"] * 100))
ax[0].set(xlabel="time (min)", ylabel=r"$\Delta R/R_0$ (%)"); ax[0].legend()
rate = pd.DataFrame(rows).set_index("jj")
if len(rate) >= 2 and (rate.rate_pct_per_min > 0).all():
    k, c = np.polyfit(rate.V_A, np.log(rate.rate_pct_per_min), 1)
    print(f"V_0 = {1e3 / k:.0f} mV  (rate x{np.exp(0.2 * k):.1f} per +200 mV)")
    ax[1].semilogy(rate.V_A, rate.rate_pct_per_min, "o")
    vv = np.linspace(rate.V_A.min(), rate.V_A.max(), 50)
    ax[1].semilogy(vv, np.exp(c + k * vv), "-")
    ax[1].set(xlabel=r"$V_A$ (V)", ylabel="initial rate (%/min)")
plt.tight_layout()
rate

In [ ]:
cal = pd.DataFrame(CAL)
assert len(cal), "no calibration pairs yet: run section 8 (tune_and_hold) first"
cal["x"] = cal.R_active / cal.R0 - 1
cal["y"] = cal.R_30min / cal.R0 - 1
if len(cal) >= 3:
    s, delta = np.polyfit(cal.x, cal.y, 1)
else:                                    # too few points: slope 1, offset = mean relaxation
    s, delta = 1.0, float((cal.y - cal.x).mean())
cal["resid_pct"] = (cal.y - (s * cal.x + delta)) * 100
print(f"stop rule: s = {s:.3f}, delta = {delta * 100:.2f} % of R0, "
      f"residual sd = {cal.resid_pct.std():.3f} %  (n = {len(cal)})")


def R_stop_for(R0, R_T):
    return R0 * (1 + (R_T / R0 - 1 - delta) / s)

cal

In [ ]:
d = log_relax6.df[log_relax6.df.event == "iv"]
d = d[d.t > 0]
fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogx(d.t / 60, (d.R / r6["R0"] - 1) * 100, ".", ms=3)
ax.axvline(30, ls=":", c="k")
ax.set(xlabel="time since last pulse (min)", ylabel=r"$\Delta R/R_0$ (%)", title="JJ06 relaxation")

## 11. Targeting trial — JJ07, JJ08
Choose a target, stop at `R_stop_for(R0, R_T)`, hold 30 min, and compare with $R_T$.

In [ ]:
TARGET_FRAC = 0.08                     # aim for R_T = 1.08 R0
TRIAL = {}
for jj in ("JJ07",):                   # land each junction, then run for that one
    R0, _ = smu.iv()
    R_T = R0 * (1 + TARGET_FRAC)
    r = tune(jj, V_A_RUN, R_stop=R_stop_for(R0, R_T), t_max=3600)
    if not r["reason"].startswith("ABORT"):
        _, R30 = hold(jj, r["t_stop"], R0)
        TRIAL[jj] = dict(R0=R0, R_T=R_T, R_30min=R30, error_pct=(R30 / R_T - 1) * 100)
pd.DataFrame(TRIAL).T

## 12. Shut down

In [45]:
smu.close()   # 0 V, output off